In [13]:
!pip install -q pandas numpy

In [14]:
import re
import math
import pandas as pd
import numpy as np

from collections import Counter, defaultdict
from pathlib import Path
from IPython.display import display


In [15]:
from google.colab import files

uploaded = files.upload()

reddit_file = "dog_whistle_reddit_sample_5000.csv"

df = pd.read_csv(reddit_file, encoding="utf-8-sig")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

display(df.head())

Saving dog_whistle_reddit_sample_5000.csv to dog_whistle_reddit_sample_5000 (1).csv
Dataset loaded successfully!
Shape: (5000, 6)

Columns: ['content', 'subreddit', 'dog_whistle', 'date', 'ingroup', 'source']


,content,subreddit,dog_whistle,date,ingroup,source
0,"politically correct power grab (which , amusin...",MGTOW,politically correct,07-01-2019,racist,Pushift 2005-2022
1,Why do they take other people’s mere existence...,GenderCynical,GC,08/22/2018,transphobic,Pushift 2005-2022
2,&gt; Let's remember that this **includes our T...,GenderCynical,TIF,04-07-2020,transphobic,Pushift 2005-2022
3,The gay Mafia. It’s all about societal degrada...,MGTOW,public schools,04-05-2020,racist,Pushift 2005-2022
4,Single mother - A mom who prioritises her own ...,MGTOW,single mother,03/17/2020,racist,Pushift 2005-2022


In [16]:

import re
import html
import unicodedata

# Use the original 5,000-post dataset
df_original = df.copy()
# Keep Unicode words, contractions, hyphenated words and numbers
TOKEN_PATTERN = re.compile(
    r"[^\W\d_]+(?:[-'’][^\W\d_]+)*|\d+(?:[.,]\d+)?%?",
    flags=re.UNICODE
)

SENTENCE_SPLIT_PATTERN = re.compile(
    r"(?<=[.!?])\s+|\n+"
)

def tokenize_comment(text):

    if pd.isna(text):
        return []

    text = str(text)

    # Decode HTML entities, including double-encoded entities
    for _ in range(3):
        decoded = html.unescape(text)
        if decoded == text:
            break
        text = decoded

    # Replace invisible formatting characters with spaces
    text = re.sub(
        r"[\u200b-\u200f\u2060\ufeff]",
        " ",
        text
    )

    # Unicode normalisation and lowercase conversion
    text = unicodedata.normalize("NFKC", text)
    text = text.casefold()

    sentences = SENTENCE_SPLIT_PATTERN.split(text)

    tokenized_sentences = []

    for sentence in sentences:
        tokens = TOKEN_PATTERN.findall(sentence)

        if tokens:
            tokenized_sentences.append(tokens)

    return tokenized_sentences


# IMPORTANT: start with a fresh list
documents = []

for comment in df["content"]:
    documents.append(tokenize_comment(comment))

print("Tokenization completed!")
print("Number of tokenized documents:", len(documents))


Tokenization completed!
Number of tokenized documents: 5000


In [17]:
#Extract bigrams and trigrams
ngram_counts = {
    2: Counter(),
    3: Counter()
}

document_counts = {
    2: Counter(),
    3: Counter()
}

# Marginal counts for association calculations
left_counts = Counter()
right_counts = Counter()

trigram_prefix_counts = Counter()
trigram_next_counts = Counter()

total_bigrams = 0
total_trigrams = 0

# Keep example comments for each expression
examples = {
    2: defaultdict(list),
    3: defaultdict(list)
}

for doc_id, sentences in enumerate(documents):

    document_ngrams = {
        2: set(),
        3: set()
    }

    for tokens in sentences:

        # Unigram counts are used to calculate bigram marginals
        for i in range(len(tokens) - 1):
            left_counts[tokens[i]] += 1
            right_counts[tokens[i + 1]] += 1

            bigram = tuple(tokens[i:i+2])
            ngram_counts[2][bigram] += 1
            document_ngrams[2].add(bigram)

            total_bigrams += 1

            if len(examples[2][bigram]) < 5:
                examples[2][bigram].append(
                    df.iloc[doc_id]["content"]
                )

        # Trigram counts and marginals
        for i in range(len(tokens) - 2):

            trigram = tuple(tokens[i:i+3])
            prefix = tuple(tokens[i:i+2])
            next_word = tokens[i+2]

            ngram_counts[3][trigram] += 1
            document_ngrams[3].add(trigram)

            trigram_prefix_counts[prefix] += 1
            trigram_next_counts[next_word] += 1

            total_trigrams += 1

            if len(examples[3][trigram]) < 5:
                examples[3][trigram].append(
                    df.iloc[doc_id]["content"]
                )

    for n in (2, 3):
        document_counts[n].update(document_ngrams[n])


print("N-gram extraction completed!")

print("Total bigram positions:", total_bigrams)
print("Total trigram positions:", total_trigrams)

print("Unique bigrams:", len(ngram_counts[2]))
print("Unique trigrams:", len(ngram_counts[3]))

N-gram extraction completed!
Total bigram positions: 204997
Total trigram positions: 191634
Unique bigrams: 103927
Unique trigrams: 168311


In [18]:
#Apply min freq filtering
MIN_FREQUENCY = 5

filtered_ngrams = {
    2: {
        term: count
        for term, count in ngram_counts[2].items()
        if count >= MIN_FREQUENCY
    },

    3: {
        term: count
        for term, count in ngram_counts[3].items()
        if count >= MIN_FREQUENCY
    }
}

print("Minimum occurrence threshold:", MIN_FREQUENCY)

print("Bigrams remaining:", len(filtered_ngrams[2]))
print("Trigrams remaining:", len(filtered_ngrams[3]))

Minimum occurrence threshold: 5
Bigrams remaining: 5598
Trigrams remaining: 1318


In [19]:

thresholds = [5, 10, 20]

threshold_results = {}

for threshold in thresholds:

    bigrams_filtered = {
        term: count
        for term, count in ngram_counts[2].items()
        if count >= threshold
    }

    trigrams_filtered = {
        term: count
        for term, count in ngram_counts[3].items()
        if count >= threshold
    }

    threshold_results[threshold] = {
        "bigrams": bigrams_filtered,
        "trigrams": trigrams_filtered
    }

    print(f"\nMinimum frequency: {threshold}")
    print("Bigrams remaining:", len(bigrams_filtered))
    print("Trigrams remaining:", len(trigrams_filtered))



Minimum frequency: 5
Bigrams remaining: 5598
Trigrams remaining: 1318

Minimum frequency: 10
Bigrams remaining: 2230
Trigrams remaining: 276

Minimum frequency: 20
Bigrams remaining: 889
Trigrams remaining: 57


In [20]:
def calculate_pmi(joint_count, left_count, right_count, total):
    """
    Calculate Pointwise Mutual Information (PMI).
    Uses log base 2.
    """

    if joint_count == 0 or left_count == 0 or right_count == 0 or total == 0:
        return np.nan

    p_joint = joint_count / total
    p_left = left_count / total
    p_right = right_count / total

    return math.log2(
        p_joint / (p_left * p_right)
    )


bigram_results = []

for bigram, count in filtered_ngrams[2].items():

    word1, word2 = bigram

    pmi = calculate_pmi(
        count,
        left_counts[word1],
        right_counts[word2],
        total_bigrams
    )

    bigram_results.append({
        "expression": " ".join(bigram),
        "occurrence_frequency": count,
        "document_frequency": document_counts[2][bigram],
        "PMI": pmi
    })


trigram_results = []

for trigram, count in filtered_ngrams[3].items():

    prefix = tuple(trigram[:2])
    next_word = trigram[2]

    pmi = calculate_pmi(
        count,
        trigram_prefix_counts[prefix],
        trigram_next_counts[next_word],
        total_trigrams
    )

    trigram_results.append({
        "expression": " ".join(trigram),
        "occurrence_frequency": count,
        "document_frequency": document_counts[3][trigram],
        "PMI": pmi
    })


pmi_bigram_df = pd.DataFrame(bigram_results)
pmi_trigram_df = pd.DataFrame(trigram_results)

pmi_bigram_df = pmi_bigram_df.sort_values(
    "PMI", ascending=False
).reset_index(drop=True)

pmi_trigram_df = pmi_trigram_df.sort_values(
    "PMI", ascending=False
).reset_index(drop=True)

print("PMI calculation completed!")

print("\nTop 20 bigrams by PMI:")
display(pmi_bigram_df.head(20))

print("\nTop 20 trigrams by PMI:")
display(pmi_trigram_df.head(20))

PMI calculation completed!

Top 20 bigrams by PMI:


,expression,occurrence_frequency,document_frequency,PMI
0,harry potter,6,6,15.060281
1,bernie sanders,7,6,14.645243
2,vice versa,5,5,14.645243
3,honk honk,5,5,14.574854
4,bodily autonomy,9,8,13.944804
5,la la,5,1,13.627321
6,rabbit hole,5,5,13.574854
7,mentally ill,10,9,13.507740
8,virtue signaling,6,5,13.422851
9,george soros,5,5,13.337815



Top 20 trigrams by PMI:


,expression,occurrence_frequency,document_frequency,PMI
0,along the lines,5,5,13.740639
1,friend of mine,7,7,12.963032
2,pump and dump,10,10,12.951059
3,an echo chamber,5,5,12.825528
4,quick google search,5,5,12.690013
5,fucks beta bucks,6,6,12.681745
6,in public schools,5,5,12.530072
7,chad and tyrone,6,6,12.503600
8,the front page,6,6,12.439470
9,the middle east,6,6,12.319175


In [21]:
def calculate_llr(k11, row_total, col_total, total):
    """
    Calculate the G-test log-likelihood ratio (G²)
    from a 2x2 contingency table.
    """

    if total <= 0:
        return np.nan

    k12 = row_total - k11
    k21 = col_total - k11
    k22 = total - k11 - k12 - k21

    observed = np.array([
        [k11, k12],
        [k21, k22]
    ], dtype=float)

    row_sums = observed.sum(axis=1)
    col_sums = observed.sum(axis=0)

    expected = np.outer(row_sums, col_sums) / total

    mask = observed > 0

    llr = 2 * np.sum(
        observed[mask] * np.log(
            observed[mask] / expected[mask]
        )
    )

    return float(llr)


# Bigram LLR
llr_bigram_results = []

for bigram, count in filtered_ngrams[2].items():

    word1, word2 = bigram

    llr = calculate_llr(
        count,
        left_counts[word1],
        right_counts[word2],
        total_bigrams
    )

    llr_bigram_results.append({
        "expression": " ".join(bigram),
        "occurrence_frequency": count,
        "document_frequency": document_counts[2][bigram],
        "LLR": llr
    })


# Trigram LLR: two-word prefix versus the following word
llr_trigram_results = []

for trigram, count in filtered_ngrams[3].items():

    prefix = tuple(trigram[:2])
    next_word = trigram[2]

    llr = calculate_llr(
        count,
        trigram_prefix_counts[prefix],
        trigram_next_counts[next_word],
        total_trigrams
    )

    llr_trigram_results.append({
        "expression": " ".join(trigram),
        "occurrence_frequency": count,
        "document_frequency": document_counts[3][trigram],
        "LLR": llr
    })


llr_bigram_df = pd.DataFrame(llr_bigram_results)
llr_trigram_df = pd.DataFrame(llr_trigram_results)

llr_bigram_df = llr_bigram_df.sort_values(
    "LLR", ascending=False
).reset_index(drop=True)

llr_trigram_df = llr_trigram_df.sort_values(
    "LLR", ascending=False
).reset_index(drop=True)

print("LLR calculation completed!")

print("\nTop 20 bigrams by LLR:")
display(llr_bigram_df.head(20))

print("\nTop 20 trigrams by LLR:")
display(llr_trigram_df.head(20))

LLR calculation completed!

Top 20 bigrams by LLR:


,expression,occurrence_frequency,document_frequency,LLR
0,based on,328,318,2714.291947
1,to be,515,471,1912.819218
2,blue pill,135,132,1501.690288
3,a lot,195,187,1356.243329
4,clown world,99,98,1340.856882
5,the same,231,217,1324.141573
6,in the,570,521,1285.939691
7,gender critical,103,102,1274.592362
8,blue pilled,101,99,1208.000257
9,a single,331,317,1196.725315



Top 20 trigrams by LLR:


,expression,occurrence_frequency,document_frequency,LLR
0,a lot of,135,130,832.321865
1,a single mom,52,50,506.169939
2,a single mother,51,51,486.243701
3,the blue pill,31,30,354.094273
4,the red pill,22,21,296.581215
5,i don't know,38,37,295.171062
6,at this point,22,22,271.889619
7,is based on,33,33,271.436976
8,the fact that,37,36,267.567076
9,when it comes,17,17,251.307106


In [22]:

def combine_results(n):

    if n == 2:
        pmi_df = pmi_bigram_df.copy()
        llr_df = llr_bigram_df.copy()
        counts = filtered_ngrams[2]
        docs = document_counts[2]

    else:
        pmi_df = pmi_trigram_df.copy()
        llr_df = llr_trigram_df.copy()
        counts = filtered_ngrams[3]
        docs = document_counts[3]

    # Frequency ranking
    frequency_df = pd.DataFrame([
        {
            "expression": " ".join(term),
            "occurrence_frequency": count,
            "document_frequency": docs[term]
        }
        for term, count in counts.items()
    ])

    frequency_df = frequency_df.sort_values(
        "occurrence_frequency", ascending=False
    ).reset_index(drop=True)

    frequency_df["frequency_rank"] = range(
        1, len(frequency_df) + 1
    )

    # PMI ranking
    pmi_df = pmi_df.sort_values(
        "PMI", ascending=False
    ).reset_index(drop=True)

    pmi_df["PMI_rank"] = range(1, len(pmi_df) + 1)

    # LLR ranking
    llr_df = llr_df.sort_values(
        "LLR", ascending=False
    ).reset_index(drop=True)

    llr_df["LLR_rank"] = range(1, len(llr_df) + 1)

    # Merge all scores
    combined = frequency_df.merge(
        pmi_df[["expression", "PMI", "PMI_rank"]],
        on="expression",
        how="left"
    )

    combined = combined.merge(
        llr_df[["expression", "LLR", "LLR_rank"]],
        on="expression",
        how="left"
    )

    return combined


combined_bigrams = combine_results(2)
combined_trigrams = combine_results(3)

print("Combined bigram results:")
display(combined_bigrams.head(20))

print("\nCombined trigram results:")
display(combined_trigrams.head(20))


Combined bigram results:


,expression,occurrence_frequency,document_frequency,frequency_rank,PMI,PMI_rank,LLR,LLR_rank
0,of the,575,516,1,2.029082,3614,827.956662,25
1,in the,570,521,2,2.672918,2915,1285.939691,7
2,to be,515,471,3,3.688429,1843,1912.819218,2
3,is a,388,361,4,2.186862,3460,618.845926,35
4,a single,331,317,5,3.657004,1872,1196.725315,10
5,based on,328,318,6,6.512175,380,2714.291947,1
6,on the,295,277,7,2.391393,3234,547.655112,44
7,this is,279,257,8,3.763747,1764,1016.546533,17
8,as a,276,254,9,3.025228,2524,734.809393,28
9,they are,268,245,10,3.689342,1842,938.963120,18



Combined trigram results:


,expression,occurrence_frequency,document_frequency,frequency_rank,PMI,PMI_rank,LLR,LLR_rank
0,a lot of,135,130,1,5.139448,600,832.321865,1
1,to be a,66,63,2,2.289970,1202,110.855919,87
2,one of the,53,52,3,3.549909,1002,186.402998,20
3,a single mom,52,50,4,7.970856,153,506.169939,2
4,a single mother,51,51,5,7.861047,156,486.243701,3
5,this is a,42,42,6,2.507637,1164,81.648525,177
6,based on the,42,42,7,1.874456,1237,51.279053,416
7,i don't know,38,37,8,6.797816,273,295.171062,6
8,the fact that,37,36,9,5.774454,449,267.567076,9
9,the rest of,33,33,10,5.428955,541,228.559733,14


In [23]:

import pandas as pd

threshold_summary = pd.DataFrame([
    {
        "minimum_frequency": threshold,
        "bigram_count": len(threshold_results[threshold]["bigrams"]),
        "trigram_count": len(threshold_results[threshold]["trigrams"])
    }
    for threshold in thresholds
])

display(threshold_summary)


,minimum_frequency,bigram_count,trigram_count
0,5,5598,1318
1,10,2230,276
2,20,889,57


In [24]:

threshold_pmi_results = {}

for threshold in thresholds:

    # Bigram PMI
    bigram_results = []

    for bigram, count in threshold_results[threshold]["bigrams"].items():

        word1, word2 = bigram

        pmi = calculate_pmi(
            count,
            left_counts[word1],
            right_counts[word2],
            total_bigrams
        )

        bigram_results.append({
            "expression": " ".join(bigram),
            "occurrence_frequency": count,
            "document_frequency": document_counts[2][bigram],
            "PMI": pmi
        })

    # Trigram PMI
    trigram_results = []

    for trigram, count in threshold_results[threshold]["trigrams"].items():

        prefix = tuple(trigram[:2])
        next_word = trigram[2]

        pmi = calculate_pmi(
            count,
            trigram_prefix_counts[prefix],
            trigram_next_counts[next_word],
            total_trigrams
        )

        trigram_results.append({
            "expression": " ".join(trigram),
            "occurrence_frequency": count,
            "document_frequency": document_counts[3][trigram],
            "PMI": pmi
        })

    # Convert to DataFrames and sort
    bigram_pmi_df = pd.DataFrame(bigram_results)
    trigram_pmi_df = pd.DataFrame(trigram_results)

    bigram_pmi_df = bigram_pmi_df.sort_values(
        "PMI", ascending=False
    ).reset_index(drop=True)

    trigram_pmi_df = trigram_pmi_df.sort_values(
        "PMI", ascending=False
    ).reset_index(drop=True)

    threshold_pmi_results[threshold] = {
        "bigrams": bigram_pmi_df,
        "trigrams": trigram_pmi_df
    }

    print(f"\nPMI results for minimum frequency {threshold}")
    print("Bigrams:", len(bigram_pmi_df))
    print("Trigrams:", len(trigram_pmi_df))

print("\nPMI calculations completed for all thresholds!")



PMI results for minimum frequency 5
Bigrams: 5598
Trigrams: 1318

PMI results for minimum frequency 10
Bigrams: 2230
Trigrams: 276

PMI results for minimum frequency 20
Bigrams: 889
Trigrams: 57

PMI calculations completed for all thresholds!


In [25]:

for threshold in thresholds:

    print(f"\n{'='*60}")
    print(f"MINIMUM FREQUENCY: {threshold}")
    print(f"{'='*60}")

    print("\nTOP 10 BIGRAMS BY PMI")
    display(threshold_pmi_results[threshold]["bigrams"].head(10))

    print("\nTOP 10 TRIGRAMS BY PMI")
    display(threshold_pmi_results[threshold]["trigrams"].head(10))



MINIMUM FREQUENCY: 5

TOP 10 BIGRAMS BY PMI


,expression,occurrence_frequency,document_frequency,PMI
0,harry potter,6,6,15.060281
1,bernie sanders,7,6,14.645243
2,vice versa,5,5,14.645243
3,honk honk,5,5,14.574854
4,bodily autonomy,9,8,13.944804
5,la la,5,1,13.627321
6,rabbit hole,5,5,13.574854
7,mentally ill,10,9,13.507740
8,virtue signaling,6,5,13.422851
9,george soros,5,5,13.337815



TOP 10 TRIGRAMS BY PMI


,expression,occurrence_frequency,document_frequency,PMI
0,along the lines,5,5,13.740639
1,friend of mine,7,7,12.963032
2,pump and dump,10,10,12.951059
3,an echo chamber,5,5,12.825528
4,quick google search,5,5,12.690013
5,fucks beta bucks,6,6,12.681745
6,in public schools,5,5,12.530072
7,chad and tyrone,6,6,12.503600
8,the front page,6,6,12.439470
9,the middle east,6,6,12.319175



MINIMUM FREQUENCY: 10

TOP 10 BIGRAMS BY PMI


,expression,occurrence_frequency,document_frequency,PMI
0,mentally ill,10,9,13.507740
1,echo chambers,10,10,12.475318
2,echo chamber,21,21,12.408204
3,political correctness,12,11,12.397316
4,affirmative action,20,20,12.365401
5,politically correct,14,14,12.195210
6,public schools,10,10,11.121681
7,personal responsibility,18,18,11.033809
8,ah yes,15,15,11.028572
9,video games,13,9,10.995849


... (2 outputs left out)


MINIMUM FREQUENCY: 20

TOP 10 BIGRAMS BY PMI


,expression,occurrence_frequency,document_frequency,PMI
0,echo chamber,21,21,12.408204
1,affirmative action,20,20,12.365401
2,middle class,52,49,10.112663
3,white knights,20,20,9.938229
4,social media,31,28,9.621802
5,clown world,99,98,9.571994
6,beta bux,30,29,9.336904
7,years ago,32,32,9.255881
8,working class,27,24,9.220222
9,year old,28,27,9.209424



TOP 10 TRIGRAMS BY PMI


,expression,occurrence_frequency,document_frequency,PMI
0,the red pill,22,21,9.847239
1,at this point,22,22,9.605480
2,the blue pill,31,30,9.058211
3,a single mom,52,50,7.970856
4,a single mother,51,51,7.861047
5,as long as,23,23,7.339760
6,a trans woman,21,21,7.229505
7,to deal with,21,21,6.990530
8,i don't know,38,37,6.797816
9,there is no,31,31,6.731888


In [26]:

threshold_llr_results = {}

for threshold in thresholds:

    # Bigram LLR
    bigram_llr_results = []

    for bigram, count in threshold_results[threshold]["bigrams"].items():

        word1, word2 = bigram

        llr = calculate_llr(
            count,
            left_counts[word1],
            right_counts[word2],
            total_bigrams
        )

        bigram_llr_results.append({
            "expression": " ".join(bigram),
            "occurrence_frequency": count,
            "document_frequency": document_counts[2][bigram],
            "LLR": llr
        })

    # Trigram LLR
    trigram_llr_results = []

    for trigram, count in threshold_results[threshold]["trigrams"].items():

        prefix = tuple(trigram[:2])
        next_word = trigram[2]

        llr = calculate_llr(
            count,
            trigram_prefix_counts[prefix],
            trigram_next_counts[next_word],
            total_trigrams
        )

        trigram_llr_results.append({
            "expression": " ".join(trigram),
            "occurrence_frequency": count,
            "document_frequency": document_counts[3][trigram],
            "LLR": llr
        })

    # Convert to DataFrames
    bigram_llr_df = pd.DataFrame(bigram_llr_results)
    trigram_llr_df = pd.DataFrame(trigram_llr_results)

    # Sort by LLR
    bigram_llr_df = bigram_llr_df.sort_values(
        "LLR", ascending=False
    ).reset_index(drop=True)

    trigram_llr_df = trigram_llr_df.sort_values(
        "LLR", ascending=False
    ).reset_index(drop=True)

    threshold_llr_results[threshold] = {
        "bigrams": bigram_llr_df,
        "trigrams": trigram_llr_df
    }

    print(f"\nLLR results for minimum frequency {threshold}")
    print("Bigrams:", len(bigram_llr_df))
    print("Trigrams:", len(trigram_llr_df))

print("\nLLR calculations completed for all thresholds!")



LLR results for minimum frequency 5
Bigrams: 5598
Trigrams: 1318

LLR results for minimum frequency 10
Bigrams: 2230
Trigrams: 276

LLR results for minimum frequency 20
Bigrams: 889
Trigrams: 57

LLR calculations completed for all thresholds!


In [27]:

for threshold in thresholds:

    print(f"\n{'='*60}")
    print(f"MINIMUM FREQUENCY: {threshold}")
    print(f"{'='*60}")

    print("\nTOP 10 BIGRAMS BY LLR")
    display(threshold_llr_results[threshold]["bigrams"].head(10))

    print("\nTOP 10 TRIGRAMS BY LLR")
    display(threshold_llr_results[threshold]["trigrams"].head(10))



MINIMUM FREQUENCY: 5

TOP 10 BIGRAMS BY LLR


,expression,occurrence_frequency,document_frequency,LLR
0,based on,328,318,2714.291947
1,to be,515,471,1912.819218
2,blue pill,135,132,1501.690288
3,a lot,195,187,1356.243329
4,clown world,99,98,1340.856882
5,the same,231,217,1324.141573
6,in the,570,521,1285.939691
7,gender critical,103,102,1274.592362
8,blue pilled,101,99,1208.000257
9,a single,331,317,1196.725315



TOP 10 TRIGRAMS BY LLR


,expression,occurrence_frequency,document_frequency,LLR
0,a lot of,135,130,832.321865
1,a single mom,52,50,506.169939
2,a single mother,51,51,486.243701
3,the blue pill,31,30,354.094273
4,the red pill,22,21,296.581215
5,i don't know,38,37,295.171062
6,at this point,22,22,271.889619
7,is based on,33,33,271.436976
8,the fact that,37,36,267.567076
9,when it comes,17,17,251.307106



MINIMUM FREQUENCY: 10

TOP 10 BIGRAMS BY LLR


,expression,occurrence_frequency,document_frequency,LLR
0,based on,328,318,2714.291947
1,to be,515,471,1912.819218
2,blue pill,135,132,1501.690288
3,a lot,195,187,1356.243329
4,clown world,99,98,1340.856882
5,the same,231,217,1324.141573
6,in the,570,521,1285.939691
7,gender critical,103,102,1274.592362
8,blue pilled,101,99,1208.000257
9,a single,331,317,1196.725315


... (2 outputs left out)


MINIMUM FREQUENCY: 20

TOP 10 BIGRAMS BY LLR


,expression,occurrence_frequency,document_frequency,LLR
0,based on,328,318,2714.291947
1,to be,515,471,1912.819218
2,blue pill,135,132,1501.690288
3,a lot,195,187,1356.243329
4,clown world,99,98,1340.856882
5,the same,231,217,1324.141573
6,in the,570,521,1285.939691
7,gender critical,103,102,1274.592362
8,blue pilled,101,99,1208.000257
9,a single,331,317,1196.725315



TOP 10 TRIGRAMS BY LLR


,expression,occurrence_frequency,document_frequency,LLR
0,a lot of,135,130,832.321865
1,a single mom,52,50,506.169939
2,a single mother,51,51,486.243701
3,the blue pill,31,30,354.094273
4,the red pill,22,21,296.581215
5,i don't know,38,37,295.171062
6,at this point,22,22,271.889619
7,is based on,33,33,271.436976
8,the fact that,37,36,267.567076
9,a bunch of,31,31,242.301303


In [28]:

threshold_combined_results = {}

for threshold in thresholds:

    for n in [2, 3]:

        if n == 2:
            ngram_type = "bigrams"
            counts = threshold_results[threshold]["bigrams"]
        else:
            ngram_type = "trigrams"
            counts = threshold_results[threshold]["trigrams"]

        # Frequency information
        frequency_df = pd.DataFrame([
            {
                "expression": " ".join(term),
                "occurrence_frequency": count,
                "document_frequency": document_counts[n][term]
            }
            for term, count in counts.items()
        ])

        frequency_df = frequency_df.sort_values(
            "occurrence_frequency", ascending=False
        ).reset_index(drop=True)

        frequency_df["frequency_rank"] = range(
            1, len(frequency_df) + 1
        )

        # Get PMI and LLR results
        pmi_df = threshold_pmi_results[threshold][ngram_type].copy()
        llr_df = threshold_llr_results[threshold][ngram_type].copy()

        # Assign ranks
        pmi_df = pmi_df.sort_values(
            "PMI", ascending=False
        ).reset_index(drop=True)

        pmi_df["PMI_rank"] = range(1, len(pmi_df) + 1)

        llr_df = llr_df.sort_values(
            "LLR", ascending=False
        ).reset_index(drop=True)

        llr_df["LLR_rank"] = range(1, len(llr_df) + 1)

        # Merge all results
        combined = frequency_df.merge(
            pmi_df[["expression", "PMI", "PMI_rank"]],
            on="expression",
            how="left"
        )

        combined = combined.merge(
            llr_df[["expression", "LLR", "LLR_rank"]],
            on="expression",
            how="left"
        )

        threshold_combined_results.setdefault(threshold, {})
        threshold_combined_results[threshold][ngram_type] = combined

print("Combined results created for all thresholds!")


Combined results created for all thresholds!


In [29]:

for threshold in thresholds:

    print(f"\n{'='*60}")
    print(f"MINIMUM FREQUENCY: {threshold}")
    print(f"{'='*60}")

    print("\nTOP 10 BIGRAMS BY FREQUENCY")
    display(
        threshold_combined_results[threshold]["bigrams"]
        .sort_values("frequency_rank")
        .head(10)
    )

    print("\nTOP 10 TRIGRAMS BY FREQUENCY")
    display(
        threshold_combined_results[threshold]["trigrams"]
        .sort_values("frequency_rank")
        .head(10)
    )



MINIMUM FREQUENCY: 5

TOP 10 BIGRAMS BY FREQUENCY


,expression,occurrence_frequency,document_frequency,frequency_rank,PMI,PMI_rank,LLR,LLR_rank
0,of the,575,516,1,2.029082,3614,827.956662,25
1,in the,570,521,2,2.672918,2915,1285.939691,7
2,to be,515,471,3,3.688429,1843,1912.819218,2
3,is a,388,361,4,2.186862,3460,618.845926,35
4,a single,331,317,5,3.657004,1872,1196.725315,10
5,based on,328,318,6,6.512175,380,2714.291947,1
6,on the,295,277,7,2.391393,3234,547.655112,44
7,this is,279,257,8,3.763747,1764,1016.546533,17
8,as a,276,254,9,3.025228,2524,734.809393,28
9,they are,268,245,10,3.689342,1842,938.963120,18



TOP 10 TRIGRAMS BY FREQUENCY


,expression,occurrence_frequency,document_frequency,frequency_rank,PMI,PMI_rank,LLR,LLR_rank
0,a lot of,135,130,1,5.139448,600,832.321865,1
1,to be a,66,63,2,2.289970,1202,110.855919,87
2,one of the,53,52,3,3.549909,1002,186.402998,20
3,a single mom,52,50,4,7.970856,153,506.169939,2
4,a single mother,51,51,5,7.861047,156,486.243701,3
5,this is a,42,42,6,2.507637,1164,81.648525,177
6,based on the,42,42,7,1.874456,1237,51.279053,416
7,i don't know,38,37,8,6.797816,273,295.171062,6
8,the fact that,37,36,9,5.774454,449,267.567076,9
9,the rest of,33,33,10,5.428955,541,228.559733,14



MINIMUM FREQUENCY: 10

TOP 10 BIGRAMS BY FREQUENCY


,expression,occurrence_frequency,document_frequency,frequency_rank,PMI,PMI_rank,LLR,LLR_rank
0,of the,575,516,1,2.029082,1457,827.956662,25
1,in the,570,521,2,2.672918,1148,1285.939691,7
2,to be,515,471,3,3.688429,695,1912.819218,2
3,is a,388,361,4,2.186862,1386,618.845926,35
4,a single,331,317,5,3.657004,710,1196.725315,10
5,based on,328,318,6,6.512175,124,2714.291947,1
6,on the,295,277,7,2.391393,1283,547.655112,44
7,this is,279,257,8,3.763747,669,1016.546533,17
8,as a,276,254,9,3.025228,985,734.809393,28
9,they are,268,245,10,3.689342,694,938.963120,18


... (2 outputs left out)


MINIMUM FREQUENCY: 20

TOP 10 BIGRAMS BY FREQUENCY


,expression,occurrence_frequency,document_frequency,frequency_rank,PMI,PMI_rank,LLR,LLR_rank
0,of the,575,516,1,2.029082,583,827.956662,25
1,in the,570,521,2,2.672918,457,1285.939691,7
2,to be,515,471,3,3.688429,273,1912.819218,2
3,is a,388,361,4,2.186862,556,618.845926,35
4,a single,331,317,5,3.657004,280,1196.725315,10
5,based on,328,318,6,6.512175,45,2714.291947,1
6,on the,295,277,7,2.391393,513,547.655112,44
7,this is,279,257,8,3.763747,260,1016.546533,17
8,as a,276,254,9,3.025228,393,734.809393,28
9,they are,268,245,10,3.689342,272,938.963120,18



TOP 10 TRIGRAMS BY FREQUENCY


,expression,occurrence_frequency,document_frequency,frequency_rank,PMI,PMI_rank,LLR,LLR_rank
0,a lot of,135,130,1,5.139448,22,832.321865,1
1,to be a,66,63,2,2.289970,53,110.855919,33
2,one of the,53,52,3,3.549909,41,186.402998,16
3,a single mom,52,50,4,7.970856,4,506.169939,2
4,a single mother,51,51,5,7.861047,5,486.243701,3
5,based on the,42,42,6,1.874456,55,51.279053,51
6,this is a,42,42,7,2.507637,49,81.648525,43
7,i don't know,38,37,8,6.797816,9,295.171062,6
8,the fact that,37,36,9,5.774454,16,267.567076,9
9,the rest of,33,33,10,5.428955,21,228.559733,13


In [30]:
#Compare top candidates across thresholds


def compare_top_candidates(ngram_type, metric, top_n=20):

    comparison_data = []

    for threshold in thresholds:

        result_df = threshold_combined_results[threshold][ngram_type]

        top_df = result_df.nlargest(top_n, metric)

        for _, row in top_df.iterrows():

            comparison_data.append({
                "threshold": threshold,
                "expression": row["expression"],
                "occurrence_frequency": row["occurrence_frequency"],
                metric: row[metric]
            })

    comparison_df = pd.DataFrame(comparison_data)

    stability = (
        comparison_df.groupby("expression")
        .agg(
            threshold_count=("threshold", "nunique"),
            thresholds_found=("threshold", lambda x: sorted(x.tolist())),
            total_occurrences=("occurrence_frequency", "max")
        )
        .reset_index()
        .sort_values(
            ["threshold_count", "total_occurrences"],
            ascending=[False, False]
        )
    )

    return comparison_df, stability


In [31]:

# Bigram comparisons
bigram_pmi_comparison, bigram_pmi_stability = compare_top_candidates(
    "bigrams", "PMI"
)

bigram_llr_comparison, bigram_llr_stability = compare_top_candidates(
    "bigrams", "LLR"
)

# Trigram comparisons
trigram_pmi_comparison, trigram_pmi_stability = compare_top_candidates(
    "trigrams", "PMI"
)

trigram_llr_comparison, trigram_llr_stability = compare_top_candidates(
    "trigrams", "LLR"
)

print("\nTOP BIGRAMS BY PMI – STABILITY")
display(bigram_pmi_stability.head(20))

print("\nTOP BIGRAMS BY LLR – STABILITY")
display(bigram_llr_stability.head(20))

print("\nTOP TRIGRAMS BY PMI – STABILITY")
display(trigram_pmi_stability.head(20))

print("\nTOP TRIGRAMS BY LLR – STABILITY")
display(trigram_llr_stability.head(20))



TOP BIGRAMS BY PMI – STABILITY


,expression,threshold_count,thresholds_found,total_occurrences
12,clown world,2,"[10, 20]",99
33,middle class,2,"[10, 20]",52
42,social media,2,"[10, 20]",31
20,echo chamber,2,"[10, 20]",21
0,affirmative action,2,"[10, 20]",20
49,white knights,2,"[10, 20]",20
32,mentally ill,2,"[5, 10]",10
7,blue pill,1,[20],135
24,gender critical,1,[20],103
8,blue pilled,1,[20],101



TOP BIGRAMS BY LLR – STABILITY


,expression,threshold_count,thresholds_found,total_occurrences
12,in the,3,"[5, 10, 20]",570
17,to be,3,"[5, 10, 20]",515
1,a single,3,"[5, 10, 20]",331
3,based on,3,"[5, 10, 20]",328
16,this is,3,"[5, 10, 20]",279
15,they are,3,"[5, 10, 20]",268
11,if you,3,"[5, 10, 20]",249
14,the same,3,"[5, 10, 20]",231
19,want to,3,"[5, 10, 20]",216
0,a lot,3,"[5, 10, 20]",195



TOP TRIGRAMS BY PMI – STABILITY


,expression,threshold_count,thresholds_found,total_occurrences
36,the blue pill,2,"[10, 20]",31
11,at this point,2,"[10, 20]",22
44,the red pill,2,"[10, 20]",22
17,hit the wall,2,"[5, 10]",10
32,pump and dump,2,"[5, 10]",10
3,a single mom,1,[20],52
4,a single mother,1,[20],51
18,i don't know,1,[20],38
37,the fact that,1,[20],37
26,is based on,1,[20],33



TOP TRIGRAMS BY LLR – STABILITY


,expression,threshold_count,thresholds_found,total_occurrences
1,a lot of,3,"[5, 10, 20]",135
12,one of the,3,"[5, 10, 20]",53
2,a single mom,3,"[5, 10, 20]",52
3,a single mother,3,"[5, 10, 20]",51
9,i don't know,3,"[5, 10, 20]",38
14,the fact that,3,"[5, 10, 20]",37
10,is based on,3,"[5, 10, 20]",33
18,the rest of,3,"[5, 10, 20]",33
0,a bunch of,3,"[5, 10, 20]",31
13,the blue pill,3,"[5, 10, 20]",31


In [32]:

def get_consistent_candidates(stability_df):

    consistent = stability_df[
        stability_df["threshold_count"] == 3
    ].copy()

    return consistent.sort_values(
        "total_occurrences", ascending=False
    ).reset_index(drop=True)


consistent_candidates = {
    "bigram_pmi": get_consistent_candidates(bigram_pmi_stability),
    "bigram_llr": get_consistent_candidates(bigram_llr_stability),
    "trigram_pmi": get_consistent_candidates(trigram_pmi_stability),
    "trigram_llr": get_consistent_candidates(trigram_llr_stability)
}

for name, result_df in consistent_candidates.items():

    print(f"\n{'='*55}")
    print(name.upper())
    print(f"{'='*55}")

    display(result_df)



BIGRAM_PMI


,expression,threshold_count,thresholds_found,total_occurrences



BIGRAM_LLR


,expression,threshold_count,thresholds_found,total_occurrences
0,in the,3,"[5, 10, 20]",570
1,to be,3,"[5, 10, 20]",515
2,a single,3,"[5, 10, 20]",331
3,based on,3,"[5, 10, 20]",328
4,this is,3,"[5, 10, 20]",279
5,they are,3,"[5, 10, 20]",268
6,if you,3,"[5, 10, 20]",249
7,the same,3,"[5, 10, 20]",231
8,want to,3,"[5, 10, 20]",216
9,a lot,3,"[5, 10, 20]",195



TRIGRAM_PMI


,expression,threshold_count,thresholds_found,total_occurrences



TRIGRAM_LLR


,expression,threshold_count,thresholds_found,total_occurrences
0,a lot of,3,"[5, 10, 20]",135
1,one of the,3,"[5, 10, 20]",53
2,a single mom,3,"[5, 10, 20]",52
3,a single mother,3,"[5, 10, 20]",51
4,i don't know,3,"[5, 10, 20]",38
5,the fact that,3,"[5, 10, 20]",37
6,is based on,3,"[5, 10, 20]",33
7,the rest of,3,"[5, 10, 20]",33
8,a bunch of,3,"[5, 10, 20]",31
9,the blue pill,3,"[5, 10, 20]",31


In [33]:
import os

output_folder = "Dog_Whistle_Experiment_Results"
os.makedirs(output_folder, exist_ok=True)

# 1. Export combined results for all thresholds
for threshold in thresholds:
    for ngram_type in ["bigrams", "trigrams"]:
        filename = f"{ngram_type}_combined_threshold_{threshold}.csv"

        threshold_combined_results[threshold][ngram_type].to_csv(
            os.path.join(output_folder, filename),
            index=False
        )

# 2. Export stability tables
stability_results = {
    "bigram_pmi_stability": bigram_pmi_stability,
    "bigram_llr_stability": bigram_llr_stability,
    "trigram_pmi_stability": trigram_pmi_stability,
    "trigram_llr_stability": trigram_llr_stability
}

for filename, result_df in stability_results.items():
    result_df.to_csv(
        os.path.join(output_folder, f"{filename}.csv"),
        index=False
    )

# 3. Export consistent candidates
for filename, result_df in consistent_candidates.items():
    result_df.to_csv(
        os.path.join(output_folder, f"{filename}_consistent.csv"),
        index=False
    )

print("Export completed!")
print("Number of files:", len(os.listdir(output_folder)))
print("Output folder:", output_folder)

Export completed!
Number of files: 14
Output folder: Dog_Whistle_Experiment_Results


In [34]:
import shutil
from google.colab import files

# Compress the results folder
zip_path = shutil.make_archive(
    "Dog_Whistle_Experiment_Results",
    "zip",
    output_folder
)

# Download the ZIP to your computer
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
#  Build candidate pool

candidate_rows = []

# 1. Collect top 15 PMI candidates at each threshold
for threshold in thresholds:
    for ngram_type in ["bigrams", "trigrams"]:

        result_df = threshold_combined_results[threshold][ngram_type]

        top_pmi = result_df.nlargest(15, "PMI")

        for _, row in top_pmi.iterrows():
            candidate_rows.append({
                "expression": row["expression"],
                "ngram_type": ngram_type,
                "metric": "PMI",
                "threshold": threshold,
                "frequency": row["occurrence_frequency"],
                "score": row["PMI"]
            })

# 2. Collect top 15 LLR candidates at threshold 20
for ngram_type in ["bigrams", "trigrams"]:

    result_df = threshold_combined_results[20][ngram_type]

    top_llr = result_df.nlargest(15, "LLR")

    for _, row in top_llr.iterrows():
        candidate_rows.append({
            "expression": row["expression"],
            "ngram_type": ngram_type,
            "metric": "LLR",
            "threshold": 20,
            "frequency": row["occurrence_frequency"],
            "score": row["LLR"]
        })

candidate_pool = pd.DataFrame(candidate_rows)

# Remove duplicate entries within the same metric and threshold
candidate_pool = candidate_pool.drop_duplicates(
    subset=["expression", "ngram_type", "metric", "threshold"]
)

print("Candidate pool created!")
print("Total candidate records:", len(candidate_pool))
print("Unique expressions:", candidate_pool["expression"].nunique())

display(candidate_pool.head(20))

Candidate pool created!
Total candidate records: 120
Unique expressions: 98


,expression,ngram_type,metric,threshold,frequency,score
0,harry potter,bigrams,PMI,5,6,15.060281
1,bernie sanders,bigrams,PMI,5,7,14.645243
2,vice versa,bigrams,PMI,5,5,14.645243
3,honk honk,bigrams,PMI,5,5,14.574854
4,bodily autonomy,bigrams,PMI,5,9,13.944804
5,la la,bigrams,PMI,5,5,13.627321
6,rabbit hole,bigrams,PMI,5,5,13.574854
7,mentally ill,bigrams,PMI,5,10,13.507740
8,virtue signaling,bigrams,PMI,5,6,13.422851
9,george soros,bigrams,PMI,5,5,13.337815


In [36]:
# Summarise candidate evidence

candidate_summary = (
    candidate_pool
    .groupby(["expression", "ngram_type"])
    .agg(
        metrics_found=("metric", lambda x: ", ".join(sorted(set(x)))),
        thresholds_found=("threshold", lambda x: sorted(set(x))),
        max_frequency=("frequency", "max"),
        number_of_records=("metric", "count")
    )
    .reset_index()
)

candidate_summary = candidate_summary.sort_values(
    ["number_of_records", "max_frequency"],
    ascending=[False, False]
).reset_index(drop=True)

print("Candidate summary:")
print("Unique expression/type combinations:", len(candidate_summary))

display(candidate_summary)

Candidate summary:
Unique expression/type combinations: 98


,expression,ngram_type,metrics_found,thresholds_found,max_frequency,number_of_records
0,at this point,trigrams,"LLR, PMI","[10, 20]",22,3
1,the red pill,trigrams,"LLR, PMI","[10, 20]",22,3
2,gender critical,bigrams,"LLR, PMI",[20],103,2
3,blue pilled,bigrams,"LLR, PMI",[20],101,2
4,clown world,bigrams,"LLR, PMI",[20],99,2
...,...,...,...,...,...,...
93,la la,bigrams,PMI,[5],5,1
94,quick google search,trigrams,PMI,[5],5,1
95,rabbit hole,bigrams,PMI,[5],5,1
96,vice versa,bigrams,PMI,[5],5,1


In [37]:
#  Export candidate selection results

candidate_pool.to_csv(
    os.path.join(output_folder, "candidate_pool.csv"),
    index=False
)

candidate_summary.to_csv(
    os.path.join(output_folder, "candidate_summary.csv"),
    index=False
)

print("Candidate files saved successfully!")

Candidate files saved successfully!


In [39]:
from google.colab import files
import os

# Correct path to files inside the results folder
files.download(os.path.join("Dog_Whistle_Experiment_Results", "candidate_pool.csv"))
files.download(os.path.join("Dog_Whistle_Experiment_Results", "candidate_summary.csv"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Matching with the glossary

In [41]:
from google.colab import files
import pandas as pd

uploaded_dictionary = files.upload()

Saving silent_signals_reference_dictionary.csv to silent_signals_reference_dictionary.csv


In [42]:
import io

dictionary_filename = list(uploaded_dictionary.keys())[0]

silent_signals_df = pd.read_csv(
    io.BytesIO(uploaded_dictionary[dictionary_filename])
)

print("Dictionary loaded successfully!")
print("Shape:", silent_signals_df.shape)
print("\nColumns:")
print(silent_signals_df.columns.tolist())

display(silent_signals_df.head())

Dictionary loaded successfully!
Shape: (706, 4)

Columns:
['dog_whistle', 'dog_whistle_root', 'definition', 'ingroup']


,dog_whistle,dog_whistle_root,definition,ingroup
0,SJW,social justice warrior,liberals,anti-liberal
1,SJWs,social justice warrior,liberals,anti-liberal
2,blue,blueish,Jewish,antisemitic
3,LGB,LGB rights,"Lesbian, gay, and bisexual, but not trans, rights",transphobic
4,working class,working class,white working class,racist


In [44]:
import re
import html
import unicodedata
import pandas as pd

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Decode HTML entities
    for _ in range(3):
        decoded = html.unescape(text)
        if decoded == text:
            break
        text = decoded

    # Unicode normalisation
    text = unicodedata.normalize("NFKC", text)

    # Standardise apostrophes
    text = text.replace("’", "'").replace("‘", "'")

    # Remove zero-width characters
    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)

    # Lowercase and standardise whitespace
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)

    return text


# Create separate working copies
candidates_working = candidate_summary.copy()
dictionary_working = silent_signals_df.copy()
reddit_working = df_original.copy()

# Normalise candidate expressions
candidates_working["normalized_expression"] = (
    candidates_working["expression"].apply(normalize_text)
)

# Normalise dictionary entries
dictionary_working["normalized_dog_whistle"] = (
    dictionary_working["dog_whistle"].apply(normalize_text)
)

dictionary_working["normalized_root"] = (
    dictionary_working["dog_whistle_root"].apply(normalize_text)
)

# Normalise Reddit comments
reddit_working["normalized_content"] = (
    reddit_working["content"].apply(normalize_text)
)

print("Text normalisation completed!")

Text normalisation completed!


In [46]:
#Find dictionary matches
dictionary_matches = []

for _, candidate in candidates_working.iterrows():

    expression = candidate["normalized_expression"]

    for _, reference in dictionary_working.iterrows():

        reference_fields = [
            ("dog_whistle", reference["normalized_dog_whistle"]),
            ("dog_whistle_root", reference["normalized_root"])
        ]

        for field_name, reference_term in reference_fields:

            if not reference_term:
                continue

            match_type = None

            # Exact match
            if expression == reference_term:
                match_type = "exact_match"

            # Multiword reference phrase inside candidate
            elif " " in reference_term:
                pattern = r"(?<!\w)" + re.escape(reference_term) + r"(?!\w)"

                if re.search(pattern, expression):
                    match_type = "phrase_match"

            if match_type:
                dictionary_matches.append({
                    "expression": candidate["expression"],
                    "ngram_type": candidate["ngram_type"],
                    "matched_dictionary_field": field_name,
                    "match_type": match_type,
                    "dog_whistle": reference["dog_whistle"],
                    "dog_whistle_root": reference["dog_whistle_root"],
                    "definition": reference["definition"],
                    "ingroup": reference["ingroup"]
                })

dictionary_matches_df = pd.DataFrame(dictionary_matches)

print("Dictionary matching completed!")
print("Total dictionary matches:", len(dictionary_matches_df))

if not dictionary_matches_df.empty:
    print(
        "Unique candidate expressions matched:",
        dictionary_matches_df["expression"].nunique()
    )

    display(dictionary_matches_df.head(20))
else:
    print("No direct or phrase matches found.")

Dictionary matching completed!
Total dictionary matches: 57
Unique candidate expressions matched: 22


,expression,ngram_type,matched_dictionary_field,match_type,dog_whistle,dog_whistle_root,definition,ingroup
0,gender critical,bigrams,dog_whistle_root,exact_match,GC,gender critical,Self-referential term for transphobic people,transphobic
1,gender critical,bigrams,dog_whistle_root,exact_match,gender crit,gender critical,Self-referential term for transphobic people,transphobic
2,gender critical,bigrams,dog_whistle_root,exact_match,gender-critical feminism,gender critical,Self-referential term for transphobic people,transphobic
3,clown world,bigrams,dog_whistle,exact_match,Clown World,Clown World,White supremacist signal originating in 4chan ...,white supremacist
4,clown world,bigrams,dog_whistle_root,exact_match,Clown World,Clown World,White supremacist signal originating in 4chan ...,white supremacist
5,clown world,bigrams,dog_whistle_root,exact_match,🤡 🌎,Clown World,White supremacist signal originating in 4chan ...,white supremacist
6,clown world,bigrams,dog_whistle_root,exact_match,Honkler,Clown World,White supremacist signal originating in 4chan ...,white supremacist
7,a single mom,trigrams,dog_whistle,phrase_match,single mom,single parent,Blaming Black parents (especially women) for p...,racist
8,middle class,bigrams,dog_whistle,exact_match,middle class,middle class,White middle class,racist
9,middle class,bigrams,dog_whistle_root,exact_match,middle class,middle class,White middle class,racist


In [47]:
if not dictionary_matches_df.empty:

    dictionary_match_summary = (
        dictionary_matches_df
        .groupby(["expression", "ngram_type"])
        .agg(
            dictionary_terms=(
                "dog_whistle",
                lambda x: sorted(set(x.dropna()))
            ),
            dictionary_roots=(
                "dog_whistle_root",
                lambda x: sorted(set(x.dropna()))
            ),
            definitions=(
                "definition",
                lambda x: sorted(set(x.dropna()))
            ),
            ingroups=(
                "ingroup",
                lambda x: sorted(set(x.dropna()))
            ),
            match_types=(
                "match_type",
                lambda x: sorted(set(x))
            )
        )
        .reset_index()
    )

else:
    dictionary_match_summary = pd.DataFrame(
        columns=[
            "expression", "ngram_type", "dictionary_terms",
            "dictionary_roots", "definitions", "ingroups",
            "match_types"
        ]
    )


candidate_reference_summary = candidates_working.merge(
    dictionary_match_summary,
    on=["expression", "ngram_type"],
    how="left"
)

candidate_reference_summary["reference_match"] = (
    candidate_reference_summary["dictionary_terms"].notna()
)

candidate_reference_summary["reference_match"] = (
    candidate_reference_summary["reference_match"].map(
        {True: "MATCH", False: "NO MATCH"}
    )
)

print("Candidate reference summary created!")

print("\nReference match counts:")
print(candidate_reference_summary["reference_match"].value_counts())

display(
    candidate_reference_summary[
        candidate_reference_summary["reference_match"] == "MATCH"
    ]
)

Candidate reference summary created!

Reference match counts:
reference_match
NO MATCH    76
MATCH       22
Name: count, dtype: int64


,expression,ngram_type,metrics_found,thresholds_found,max_frequency,number_of_records,normalized_expression,dictionary_terms,dictionary_roots,definitions,ingroups,match_types,reference_match
2,gender critical,bigrams,"LLR, PMI",[20],103,2,gender critical,"[GC, gender crit, gender-critical feminism]",[gender critical],[Self-referential term for transphobic people],[transphobic],[exact_match],MATCH
4,clown world,bigrams,"LLR, PMI",[20],99,2,clown world,"[Clown World, Honkler, 🤡 🌎]",[Clown World],[White supremacist signal originating in 4chan...,[white supremacist],[exact_match],MATCH
5,a single mom,trigrams,"LLR, PMI",[20],52,2,a single mom,[single mom],[single parent],[Blaming Black parents (especially women) for ...,[racist],[phrase_match],MATCH
6,middle class,bigrams,PMI,"[10, 20]",52,2,middle class,"[middle class, middle class Americans, middle-...",[middle class],[White middle class],[racist],[exact_match],MATCH
7,a single mother,trigrams,"LLR, PMI",[20],51,2,a single mother,[single mother],[single parent],[Blaming Black parents (especially women) for ...,[racist],[phrase_match],MATCH
16,affirmative action,bigrams,PMI,"[10, 20]",20,2,affirmative action,[affirmative action],[affirmative action],[White people are victims of racial mistreatment],[racist],[exact_match],MATCH
43,working class,bigrams,PMI,[20],27,1,working class,"[working class, working class communities, wor...",[working class],[white working class],[racist],[exact_match],MATCH
48,personal responsibility,bigrams,PMI,[10],18,1,personal responsibility,"[individual responsibility, personal responsib...",[personal responsibility],[Opposition to public spending perceived to be...,[racist],[exact_match],MATCH
51,big pharma,bigrams,PMI,[10],15,1,big pharma,[big pharma],[big pharma],[Evil pharmaceutical companies sell poison for...,[anti-vax],[exact_match],MATCH
53,politically correct,bigrams,PMI,[10],14,1,politically correct,[politically correct],[political correctness],[We should let people be openly racist],[racist],[exact_match],MATCH


In [48]:
context_rows = []
context_counts = []

# Columns to preserve from the original Reddit dataset
metadata_columns = [
    col for col in [
        "subreddit", "dog_whistle", "date", "ingroup", "source"
    ]
    if col in reddit_working.columns
]

for _, candidate in candidates_working.iterrows():

    expression = candidate["normalized_expression"]

    if not expression:
        continue

    # Match the complete expression, not partial words
    pattern = r"(?<!\w)" + re.escape(expression) + r"(?!\w)"

    matches = reddit_working[
        reddit_working["normalized_content"].str.contains(
            pattern,
            regex=True,
            na=False
        )
    ]

    total_matches = len(matches)

    context_counts.append({
        "expression": candidate["expression"],
        "ngram_type": candidate["ngram_type"],
        "total_matching_comments": total_matches
    })

    # Retrieve up to 10 examples per candidate
    if total_matches > 0:

        examples = matches.sample(
            n=min(10, total_matches),
            random_state=42
        )

        for original_index, comment in examples.iterrows():

            row = {
                "expression": candidate["expression"],
                "ngram_type": candidate["ngram_type"],
                "original_row_index": original_index,
                "reddit_comment": comment["content"],
                "total_matching_comments": total_matches
            }

            for column in metadata_columns:
                row[column] = comment[column]

            context_rows.append(row)


reddit_contexts_df = pd.DataFrame(context_rows)
context_counts_df = pd.DataFrame(context_counts)

print("Reddit context extraction completed!")
print("Unique candidates:", len(context_counts_df))
print("Candidates with Reddit matches:",
      (context_counts_df["total_matching_comments"] > 0).sum())
print("Total example comments extracted:", len(reddit_contexts_df))

display(context_counts_df.sort_values(
    "total_matching_comments",
    ascending=False
).head(20))

Reddit context extraction completed!
Unique candidates: 98
Candidates with Reddit matches: 98
Total example comments extracted: 852


,expression,ngram_type,total_matching_comments
20,in the,bigrams,521
21,to be,bigrams,471
23,based on,bigrams,318
22,a single,bigrams,315
24,if you,bigrams,272
25,the same,bigrams,217
26,want to,bigrams,198
27,a lot,bigrams,187
28,trans people,bigrams,155
30,blue pill,bigrams,134


In [52]:
reference_columns = [
    "expression",
    "ngram_type",
    "reference_match",
    "dictionary_terms",
    "dictionary_roots",
    "definitions",
    "ingroups"
]

reddit_contexts_with_reference_df = reddit_contexts_df.merge(
    candidate_reference_summary[reference_columns],
    on=["expression", "ngram_type"],
    how="left"
)

print("Combined context and reference information!")
print("Shape:", reddit_contexts_with_reference_df.shape)


display(
    reddit_contexts_with_reference_df[
        [
            "expression",
            "reference_match",
            "ingroups",
            "subreddit",
            "reddit_comment"
        ]
    ].head(10)
)

Combined context and reference information!
Shape: (852, 15)


,expression,reference_match,ingroups,subreddit,reddit_comment
0,at this point,NO MATCH,NaN,MGTOW,"I keep pointing this out here, but herpes infe..."
1,at this point,NO MATCH,NaN,GenderCynical,That's because GC likes to make fun of men wea...
2,at this point,NO MATCH,NaN,GenderCynical,"but at this point if I thought that using ""des..."
3,at this point,NO MATCH,NaN,GenderCynical,"I’ve yet to meet a trans person (trans women, ..."
4,at this point,NO MATCH,NaN,MGTOW,"Very few Hollywood flicks avoid the ""save the ..."
5,at this point,NO MATCH,NaN,MGTOW,"Every single one of these types of women, and ..."
6,at this point,NO MATCH,NaN,MGTOW,I'm in my third fatherless teenage boy mentee ...
7,at this point,NO MATCH,NaN,GenderCynical,"The actual dictionary definition is ""adult fem..."
8,at this point,NO MATCH,NaN,GenderCynical,Had a chaser ex that lied to me for months and...
9,at this point,NO MATCH,NaN,MGTOW,It’s funny how men are worried to express this...


In [53]:
if not reddit_contexts_with_reference_df.empty:

    example_expression = (
        reddit_contexts_with_reference_df["expression"].iloc[0]
    )

    print("Candidate expression:", example_expression)

    examples = reddit_contexts_with_reference_df[
        reddit_contexts_with_reference_df["expression"] == example_expression
    ]

    for _, row in examples.iterrows():

        print("\n" + "=" * 80)
        print("Subreddit:", row.get("subreddit", "N/A"))
        print("Dictionary match:", row.get("reference_match", "N/A"))
        print("Reference ingroup:", row.get("ingroups", "N/A"))
        print("\nComment:")
        print(row["reddit_comment"])

else:
    print("No matching Reddit comments were found.")

Candidate expression: at this point

Subreddit: MGTOW
Dictionary match: NO MATCH
Reference ingroup: nan

Comment:
I keep pointing this out here, but herpes infection rates exceed 80% in some demographics. It's seen as strange to not be infected at this point by some. Truly clown world.

Subreddit: GenderCynical
Dictionary match: NO MATCH
Reference ingroup: nan

Comment:
That's because GC likes to make fun of men wearing make-up or "women's" clothes. "Man in dress" is one of their go-to insults. "Abolishing gender" has pretty much become a dogwhistle for "eradicating trans people" at this point.

Subreddit: GenderCynical
Dictionary match: NO MATCH
Reference ingroup: nan

Comment:
but at this point if I thought that using "desirability politics" instead would move the conversation forward a single inch, I'd be more than happy to.

Subreddit: GenderCynical
Dictionary match: NO MATCH
Reference ingroup: nan

Comment:
I’ve yet to meet a trans person (trans women, especially) who hasn’t been 

In [54]:
target_expression = "the red pill"

examples = reddit_contexts_with_reference_df[
    reddit_contexts_with_reference_df["expression"] == target_expression
]

print("Candidate:", target_expression)
print("Number of extracted examples:", len(examples))

for _, row in examples.iterrows():
    print("\n" + "=" * 90)
    print("Subreddit:", row.get("subreddit", "N/A"))
    print("Dictionary match:", row.get("reference_match", "N/A"))
    print("Reference ingroup:", row.get("ingroups", "N/A"))
    print("\nComment:")
    print(row["reddit_comment"])

Candidate: the red pill
Number of extracted examples: 10

Subreddit: MGTOW
Dictionary match: NO MATCH
Reference ingroup: nan

Comment:
Poor blue pilled simp, he doesn't know what to do, thats the beauty of the red pill, you know exactly what to do in these situations, she would be out on her ass in 30 minutes flat after confessing if this was me.

Subreddit: MGTOW
Dictionary match: NO MATCH
Reference ingroup: nan

Comment:
It's good you took the red pill , but you had to go through 2 heartbreaks , single mothers are not relationship material and now you understood through experience. Just have plenty of hobbies , you make a lot of friends through that , and won't feel lonely next time. Good luck

Subreddit: MGTOW
Dictionary match: NO MATCH
Reference ingroup: nan

Comment:
I think a great majority of people are blue pilled. By default So I think he meant both as to indicate the red pill Had infiltrated their communities sooner

Subreddit: MGTOW
Dictionary match: NO MATCH
Reference ingro

In [55]:
# Count matched and unmatched candidate expressions
match_counts = (
    candidate_reference_summary
    .groupby("reference_match")
    .agg(
        unique_expressions=("expression", "nunique"),
        expression_type_combinations=("expression", "count")
    )
    .reset_index()
)

display(match_counts)

# Show all dictionary-matched candidates
matched_candidates = candidate_reference_summary[
    candidate_reference_summary["reference_match"] == "MATCH"
].copy()

display(
    matched_candidates[
        [
            "expression",
            "dictionary_terms",
            "dictionary_roots",
            "definitions",
            "ingroups",
            "match_types"
        ]
    ].sort_values("expression")
)

,reference_match,unique_expressions,expression_type_combinations
0,MATCH,22,22
1,NO MATCH,76,76


,expression,dictionary_terms,dictionary_roots,definitions,ingroups,match_types
58,a clown world,"[Clown World, Honkler, 🤡 🌎]",[Clown World],[White supremacist signal originating in 4chan...,[white supremacist],[phrase_match]
5,a single mom,[single mom],[single parent],[Blaming Black parents (especially women) for ...,[racist],[phrase_match]
7,a single mother,[single mother],[single parent],[Blaming Black parents (especially women) for ...,[racist],[phrase_match]
16,affirmative action,[affirmative action],[affirmative action],[White people are victims of racial mistreatment],[racist],[exact_match]
74,bernie sanders,[Bernie Sanders],[attacks on or imagery of Jewish political lea...,[Jewish people control world politics],[antisemitic],[exact_match]
51,big pharma,[big pharma],[big pharma],[Evil pharmaceutical companies sell poison for...,[anti-vax],[exact_match]
4,clown world,"[Clown World, Honkler, 🤡 🌎]",[Clown World],[White supremacist signal originating in 4chan...,[white supremacist],[exact_match]
71,cultural marxism,[cultural Marxism],[cultural Marxism],[Jewish people are upending society],[antisemitic],[exact_match]
89,federal reserve,"[Federal Reserve, the Fed]",[Federal Reserve],[Jewish people control all the money],[antisemitic],[exact_match]
2,gender critical,"[GC, gender crit, gender-critical feminism]",[gender critical],[Self-referential term for transphobic people],[transphobic],[exact_match]


In [56]:

# Separate matched and unmatched candidates
unmatched_candidates = candidate_reference_summary[
    candidate_reference_summary["reference_match"] == "NO MATCH"
].copy()

# Prioritise candidates with strong statistical evidence
unmatched_candidates = unmatched_candidates.sort_values(
    by=["max_frequency", "number_of_records"],
    ascending=False
)

display(
    unmatched_candidates[
        [
            "expression",
            "ngram_type",
            "metrics_found",
            "thresholds_found",
            "max_frequency",
            "number_of_records"
        ]
    ].head(25)
)


,expression,ngram_type,metrics_found,thresholds_found,max_frequency,number_of_records
20,in the,bigrams,LLR,[20],570,1
21,to be,bigrams,LLR,[20],515,1
22,a single,bigrams,LLR,[20],331,1
23,based on,bigrams,LLR,[20],328,1
24,if you,bigrams,LLR,[20],249,1
25,the same,bigrams,LLR,[20],231,1
26,want to,bigrams,LLR,[20],216,1
27,a lot,bigrams,LLR,[20],195,1
28,trans people,bigrams,LLR,[20],172,1
29,a lot of,trigrams,LLR,[20],135,1


In [57]:

# Candidates selected for contextual inspection
priority_expressions = [
    "red pill",
    "blue pill",
    "blue pilled",
    "trans people",
    "single mothers",
    "child support"
]

# Retrieve up to 5 examples per expression
priority_contexts = (
    reddit_contexts_with_reference_df[
        reddit_contexts_with_reference_df["expression"].isin(
            priority_expressions
        )
    ]
    .groupby("expression", group_keys=False)
    .head(5)
)

for expression in priority_expressions:
    print("\n" + "=" * 80)
    print("EXPRESSION:", expression)
    print("=" * 80)

    examples = priority_contexts[
        priority_contexts["expression"] == expression
    ]

    if examples.empty:
        print("No context examples found.")
        continue

    for i, (_, row) in enumerate(examples.iterrows(), 1):
        print(f"\nExample {i}:")
        print("Subreddit:", row.get("subreddit", "N/A"))
        print("Comment:", row["reddit_comment"])



EXPRESSION: red pill

Example 1:
Subreddit: MGTOW
Comment: Its like in the movie Idiocracy, wise men are seen as crazy in a clown world. I once shared my mgtow views on a youtube comment section a couple days ago just to see the reactions and some were calling me a schizophrenic, they definitely had no idea what they were talking about and I was only talking about light red pill topics like about how marriage is a terrible idea and why and they were all disagreeing, it was hilarious. That is why I dont take these people seriously.

Example 2:
Subreddit: MGTOW
Comment: that's not a red pill. that's a Ostrich egg sized red pill suppository straight up the pooper. in my family. I know women who ended things after 21 and 26 years of marriage. both middle class, traditional, well educated women. there is always a ticking time bomb. some are self aware enough to know they can't improve and do stay forever, but with social media these women don't even exist anymore.

Example 3:
Subreddit: MG

In [58]:
# Display the next 20 unmatched candidates
display(
    unmatched_candidates[
        [
            "expression",
            "ngram_type",
            "metrics_found",
            "thresholds_found",
            "max_frequency",
            "number_of_records"
        ]
    ].iloc[25:45]
)

,expression,ngram_type,metrics_found,thresholds_found,max_frequency,number_of_records
40,beta bux,bigrams,PMI,[20],30,1
12,all the time,trigrams,"LLR, PMI",[20],28,2
41,not a single,trigrams,PMI,[20],28,1
42,year old,bigrams,PMI,[20],28,1
44,i don't think,trigrams,PMI,[20],24,1
13,as long as,trigrams,"LLR, PMI",[20],23,2
0,at this point,trigrams,"LLR, PMI","[10, 20]",22,3
1,the red pill,trigrams,"LLR, PMI","[10, 20]",22,3
14,echo chamber,bigrams,PMI,"[10, 20]",21,2
15,to deal with,trigrams,"LLR, PMI",[20],21,2


In [60]:
import os
import zipfile
import pandas as pd

# Create a separate folder for the final Experiment 2 results
output_folder = "Experiment_2_Final_Results"
os.makedirs(output_folder, exist_ok=True)

# All the individual DataFrames we have generated
results_to_save = {
    "candidate_summary.csv": candidate_summary,
    "candidate_reference_summary.csv": candidate_reference_summary,
    "dictionary_matches.csv": dictionary_matches_df,
    "reddit_contexts.csv": reddit_contexts_with_reference_df,
    "context_counts.csv": context_counts_df,
    "candidate_pool.csv": candidate_pool,
    "threshold_combined_results.csv": threshold_combined_results
}

# Add the stability comparison tables
safety_tables = {
    "bigram_pmi_stability.csv": bigram_pmi_stability,
    "bigram_llr_stability.csv": bigram_llr_stability,
    "trigram_pmi_stability.csv": trigram_pmi_stability,
    "trigram_llr_stability.csv": trigram_llr_stability
}
results_to_save.update(safety_tables)

# Save each standard DataFrame
for filename, dataframe in results_to_save.items():
    # If threshold_combined_results is a dict of dicts, we serialize it differently or skip direct .to_csv()
    if isinstance(dataframe, dict):
        continue
    dataframe.to_csv(
        os.path.join(output_folder, filename),
        index=False
    )

# Save consistent candidates nested dictionary DataFrames individually
for key, dataframe in consistent_candidates.items():
    dataframe.to_csv(
        os.path.join(output_folder, f"{key}_consistent_candidates.csv"),
        index=False
    )

# Save threshold_combined_results individually since it is a dictionary
for threshold, ngrams in threshold_combined_results.items():
    for ngram_type, df_ngram in ngrams.items():
        df_ngram.to_csv(
            os.path.join(output_folder, f"{ngram_type}_threshold_{threshold}.csv"),
            index=False
        )

# Package everything into one ZIP
zip_filename = "Experiment_2_Final_Results.zip"

with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for filename in os.listdir(output_folder):
        file_path = os.path.join(output_folder, filename)
        zipf.write(file_path, arcname=os.path.join(output_folder, filename))

print("Experiment 2 results saved successfully!")
print("ZIP file:", zip_filename)

# Download the ZIP
from google.colab import files
files.download(zip_filename)

Experiment 2 results saved successfully!
ZIP file: Experiment_2_Final_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>